# 02 - Feature Store Setup

This notebook covers:
1. Registering entities (CUSTOMER, TRANSACTION)
2. Creating feature views backed by Dynamic Tables
3. Inspecting feature freshness and metadata
4. Generating a point-in-time correct training dataset

**Key Concepts:**
- **Entity**: A business object with a join key (e.g., CUSTOMER_ID)
- **Feature View**: A set of features computed from source data, stored as a Dynamic Table
- **Point-in-Time Correctness**: Features retrieved as they existed at each transaction's timestamp

In [ ]:
import sys
sys.path.insert(0, "../source")
from snowpark_session import create_snowpark_session
from config import DATABASE, SCHEMA, WAREHOUSE

session = create_snowpark_session()
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()
print(f"Connected. Using warehouse: {WAREHOUSE}")

## Step 1: Register Entities

Entities define the business objects our features describe. Each entity has a unique join key that links features back to source data.

In [ ]:
from features.entities import setup_entities

fs, customer_entity, transaction_entity = setup_entities(session=session)

# List registered entities
print("\nRegistered entities:")
fs.list_entities().show()

## Step 2: Create Feature Views

We create two feature views:

1. **CUSTOMER_RISK_FEATURES** - Aggregated customer behavior:
   - Transaction velocity (count, frequency)
   - Spending patterns (avg, max, stddev)
   - Risk indicators (historical fraud rate, late-night ratio)
   - Profile enrichment (credit score, income, account age)

2. **TRANSACTION_CONTEXT_FEATURES** - Per-transaction context:
   - Amount anomaly (ratio to customer average)
   - Merchant risk signals
   - Temporal features (hour, weekend, late night)

These are backed by Dynamic Tables that auto-refresh every hour.

In [ ]:
from features.feature_views import register_feature_views

fs, customer_fv, txn_fv = register_feature_views(session=session)

# List all feature views
print("\nRegistered Feature Views:")
fs.list_feature_views().select("NAME", "VERSION", "DESC", "REFRESH_FREQ").show()

## Step 3: Inspect Feature Data

In [ ]:
# Preview customer features
print("=== CUSTOMER_RISK_FEATURES (sample) ===")
cust_fv = fs.get_feature_view("CUSTOMER_RISK_FEATURES", "V1")
cust_fv.feature_df.show(5)

print("\n=== TRANSACTION_CONTEXT_FEATURES (sample) ===")
txn_fv = fs.get_feature_view("TRANSACTION_CONTEXT_FEATURES", "V1")
txn_fv.feature_df.show(5)

## Step 4: Generate Training Dataset

The Feature Store's `generate_dataset` performs point-in-time correct feature retrieval. For each transaction in the spine, features are joined as they existed at that transaction's timestamp -- preventing future data leakage.

In [ ]:
from features.training_data import generate_training_data

training_dataset = generate_training_data(session=session)

# Preview the training data
df = training_dataset.read.to_snowpark_dataframe()
print(f"\nTraining dataset shape: {df.count()} rows")
df.show(5)